In [1]:
### RAG Pipeline _data ingestion to vector db pipe line


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_173218/4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
/home/astram/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: Dual-Domain_Explorer_Design_of_Surface-Underwater_Integrated_Robot.pdf
  ✓ Loaded 4 pages

Processing: Combining_USV_ROV_and_multimet.pdf
  ✓ Loaded 11 pages

Processing: Combined_Small-Sized_USV_and_ROV_Observation_System_for_Long-Term_Large-Scale_Spatially_Explicit_Aquatic_Monitoring.pdf
  ✓ Loaded 6 pages

Processing: Edge-Cloud_Collaborative_USV-ROV_System_Active_Tether_Control_and_Multi-Agent_Navigation_for_Marine_Pastures.pdf
  ✓ Loaded 6 pages

Processing: 1-s2.0-S0924424725009409-main.pdf
  ✓ Loaded 11 pages

Total documents loaded: 38


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 23.1.96; modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Acrobat PDFMaker 23 Word 版', 'creationdate': '2025-12-02T09:37:41+08:00', 'comments': '', 'meeting starting date': '7 Nov. 2025', 'moddate': '2026-02-19T07:23:57-05:00', 'ieee article id': '11385118', 'ieee issue id': '11384815', 'subject': '2025 5th International Conference on Artificial Intelligence, Robotics, and Communication (ICAIRC);2025; ; ;10.1109/ICAIRC68035.2025.11385118', 'sourcemodified': 'D:20251202013737', 'ieee publication id': '11382212', 'title': 'Dual-Domain Explorer: Design of Surface-Underwater Integrated Robot', 'meeting ending date': '9 Nov. 2025', 'company': '', 'source': '../data/pdf/Dual-Domain_Explorer_Design_of_Surface-Underwater_Integrated_Robot.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Dual-Domain_Explorer_Design_of

In [5]:
### Text spliiting
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAg performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators = ["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

        return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 38 documents into 235 chunks

Example chunk:
Content : Dual-Domain Explorer: Design of Surface-
Underwater Integrated Robot 
Yayue Xu1,2 
1Zhejiang University 
2Offshore Oil Engineering Co.,Ltd 
Hangzhou, China 
22460157@zju.edu.cn 
Kaipeng Ji 
Huzhou Ins...
Metadata: {'producer': 'Adobe PDF Library 23.1.96; modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Acrobat PDFMaker 23 Word 版', 'creationdate': '2025-12-02T09:37:41+08:00', 'comments': '', 'meeting starting date': '7 Nov. 2025', 'moddate': '2026-02-19T07:23:57-05:00', 'ieee article id': '11385118', 'ieee issue id': '11384815', 'subject': '2025 5th International Conference on Artificial Intelligence, Robotics, and Communication (ICAIRC);2025; ; ;10.1109/ICAIRC68035.2025.11385118', 'sourcemodified': 'D:20251202013737', 'ieee publication id': '11382212', 'title': 'Dual-Domain Explorer: Design of Surface-Underwater Integr

[Document(metadata={'producer': 'Adobe PDF Library 23.1.96; modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Acrobat PDFMaker 23 Word 版', 'creationdate': '2025-12-02T09:37:41+08:00', 'comments': '', 'meeting starting date': '7 Nov. 2025', 'moddate': '2026-02-19T07:23:57-05:00', 'ieee article id': '11385118', 'ieee issue id': '11384815', 'subject': '2025 5th International Conference on Artificial Intelligence, Robotics, and Communication (ICAIRC);2025; ; ;10.1109/ICAIRC68035.2025.11385118', 'sourcemodified': 'D:20251202013737', 'ieee publication id': '11382212', 'title': 'Dual-Domain Explorer: Design of Surface-Underwater Integrated Robot', 'meeting ending date': '9 Nov. 2025', 'company': '', 'source': '../data/pdf/Dual-Domain_Explorer_Design_of_Surface-Underwater_Integrated_Robot.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Dual-Domain_Explorer_Design_of

In [7]:
## EMBEDDING AND VECTORSTOREDB
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using sentenceTransformer"""
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
        model_name : HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesion: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model_name}: {e}")
            raise
    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts : List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape :{embeddings.shape}")
        return embeddings    
 
## intialize the embedding manager
embedding_manager =EmbeddingManager()
embedding_manager

Loading embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14156.02it/s]


Model loaded successfully. Embedding dimesion: 384


/tmp/ipykernel_173218/1086363802.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimesion: {self.model.get_sentence_embedding_dimension()}")


In [9]:
### VECTOR STORE
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


## initialize the vector store
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 235


In [10]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 23.1.96; modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Acrobat PDFMaker 23 Word 版', 'creationdate': '2025-12-02T09:37:41+08:00', 'comments': '', 'meeting starting date': '7 Nov. 2025', 'moddate': '2026-02-19T07:23:57-05:00', 'ieee article id': '11385118', 'ieee issue id': '11384815', 'subject': '2025 5th International Conference on Artificial Intelligence, Robotics, and Communication (ICAIRC);2025; ; ;10.1109/ICAIRC68035.2025.11385118', 'sourcemodified': 'D:20251202013737', 'ieee publication id': '11382212', 'title': 'Dual-Domain Explorer: Design of Surface-Underwater Integrated Robot', 'meeting ending date': '9 Nov. 2025', 'company': '', 'source': '../data/pdf/Dual-Domain_Explorer_Design_of_Surface-Underwater_Integrated_Robot.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Dual-Domain_Explorer_Design_of

In [11]:
### convert text to embeddings
texts =[doc.page_content for doc in chunks]

### generate embedding
embeddings = embedding_manager.generate_embeddings(texts)

### store in the vector database

vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 235 texts...


Batches: 100%|██████████| 8/8 [00:00<00:00, 21.18it/s]


Generated embeddings with shape :(235, 384)
Adding 235 documents to vector store...
Successfully added 235 documents to vector store
Total documents in collection: 470


In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


## initialize the RAG retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [13]:
rag_retriever.retrieve("USV is a catamaran with a waterproof electronic cabin")

Retrieving documents for query: 'USV is a catamaran with a waterproof electronic cabin'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.28it/s]

Generated embeddings with shape :(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_134554be_103',
  'content': 'keep the dynamic stability by lowering the center of mass \nbeneath the center of buoyancy [9, 10]. \n \nFigure 4 Pictures of the underwater part of the observation system. Upper: \nmodel of the ROV made by Unigraphics NX. Bottom: components of the ROV, \nincluding the camera, electronic cabin, motor (with carbon-fiber-reinforced \nprotective cages ), neutrally buoyant tether, flashlight, etc. \nB. USV \nThe main part of USV is a catamaran with a waterproof \nelectronic cabin. The catamaran can be used in most weather \nand sea conditions, which meets the requirements of long-term \nand large-scale. The two hulls and the electronic cabin are joint \ntogether by 3-mm-thick aluminum sheets. These aluminum \n“bones” have a good performance in both temper and intensity. \nThey can generate a small deformation on sudden strike like \nwinds and waves, to make an opposite torque to protect the \nbody of the vehicle. Between two hulls there is a transp

In [14]:
#### simple rag pipeline with groq llm
from langchain_groq import ChatGroq
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent / ".env" if Path.cwd().name == "notebook" else Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

### initialize the groq llm (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}

    Question: {query}
    Answer:"""

    response = llm.invoke([prompt])
    return response.content

In [15]:
import requests

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {groq_api_key}"}
)
print(response.json())

{'object': 'list', 'data': [{'id': 'canopylabs/orpheus-v1-english', 'object': 'model', 'created': 1766186316, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000, 'name': 'Canopy Labs Orpheus V1 English', 'input_modalities': ['text'], 'output_modalities': ['speech'], 'context_length': 4000, 'max_output_length': 50000, 'pricing': {'prompt': '0.000022', 'image': '0', 'request': '0', 'input_cache_read': '0.000011'}}, {'id': 'groq/compound', 'object': 'model', 'created': 1756949530, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192, 'hugging_face_id': 'groq/compound-beta', 'name': 'Compound', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 8192, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['json_mode']}, {'id': 'whisper-large-v3-turbo', 

In [16]:
Answer = rag_simple("What is unmanned surface vechile?", rag_retriever,llm)
print(Answer)

Retrieving documents for query: 'What is unmanned surface vechile?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 116.15it/s]

Generated embeddings with shape :(1, 384)
Retrieved 3 documents (after filtering)


An unmanned surface vehicle (USV) is a crew‑less boat that operates autonomously on the water’s surface. It performs tasks such as navigation, positioning, edge computing, tether tension control, and relaying communications between underwater and cloud systems, enabling long‑endurance, high‑load surface monitoring and observation.


In [17]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    context = "\n\n".join([doc['content'] for doc in results])

    def get_score(doc):
        # fall back to distance-derived score if similarity_score is somehow missing
        return doc.get('similarity_score', 1 - doc.get('distance', 1))

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': get_score(doc),
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    confidence = max(get_score(doc) for doc in results)

    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}
Answer:"""

    response = llm.invoke([prompt])

    output = {'answer': response.content, 'sources': sources, 'confidence': confidence}
    if return_context:
        output['context'] = context
    return output


result = rag_advanced("A spatially explicit, multi-depth aquatic monitoring scheme?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'A spatially explicit, multi-depth aquatic monitoring scheme?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 369.97it/s]

Generated embeddings with shape :(1, 384)
Retrieved 3 documents (after filtering)


Answer: A spatially explicit, multi‑depth aquatic monitoring scheme is a combined USV‑ROV system in which a ground control station sets GPS coordinates for sampling sites. The USV transports the ROV to each location, and the ROV, equipped with multiple sensors, dives to predetermined depths and uses closed‑loop depth control to collect water measurements. This approach allows precise, depth‑resolved sampling across a large spatial area.
Sources: [{'source': 'Combined_Small-Sized_USV_and_ROV_Observation_System_for_Long-Term_Large-Scale_Spatially_Explicit_Aquatic_Monitoring.pdf', 'page': 1, 'score': 0.3611485958099365, 'preview': 'at testifying the cooperation of these two parts and path \nguidance and navigation, will be shown in Section IV. \n \nFigu...'}, {'source': 'Combined_Small-Sized_USV_and_ROV_Observation_System_for_Long-Term_Large-Scale_Spatially_Explicit_Aquatic_Monitoring.pdf', 'page': 1, 'score': 0.3611485958099365, 'preview': 'at testifying the cooperation of these two part

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        sources = []
        summary = None
        final_answer = "No relevant context found."

        if results:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc.get('similarity_score', 1 - doc.get('distance', 1)),
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {question}
Answer:"""

            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke([prompt])
            answer = response.content

            # Add citations to answer
            citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
            final_answer = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

            # Optionally summarize answer
            if summarize:
                summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': final_answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'answer': final_answer,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }


# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is unmanned surface vehicle?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is unmanned surface vehicle?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 220.69it/s]

Generated embeddings with shape :(1, 384)
Retrieved 3 documents (after filtering)


KeyError: 'similarity_score'